In [10]:
# !pip install U langchain-huggingface lightfm langchain sentence-transformers faiss-cpu langchain_community datasets chromadb ragas

In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
# from difflib import SequenceMatcher
from datasets import load_dataset
# from lightfm import LightFM
# from lightfm.data import Dataset
# from lightfm.cross_validation import random_train_test_split
# from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

user_data = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_CDs_and_Vinyl", split="full", trust_remote_code=True)
meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_CDs_and_Vinyl", split="full", trust_remote_code=True)

def flatten_lst(col):
    # flatten lists by joining their elements with a separator (e.g., ', ')
    col['features'] = ', '.join(col['features']) if isinstance(col['features'], list) else col['features']
    col['description'] = ', '.join(col['description']) if isinstance(col['description'], list) else col['description']
    col['categories'] = ', '.join(col['categories']) if isinstance(col['categories'], list) else col['categories']

    return col

flattened_meta = meta.map(flatten_lst)

user_df = user_data.to_pandas().fillna(0)
meta_df = flattened_meta.to_pandas().fillna(0)


/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_98464/603599614.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  meta_df = flattened_meta.to_pandas().fillna(0)


In [14]:
user_df.head(5)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,True
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,True
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,True
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,True
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,False


In [16]:
user_df['verified_purchase'] = user_df['verified_purchase'].astype(int)
user_df.head(5)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,1
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,1
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,1
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,1
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,0


In [17]:
user_df.groupby(['rating','verified_purchase']).size().reset_index().pivot(columns='rating', index='verified_purchase', values=0)

rating,1.0,2.0,3.0,4.0,5.0
verified_purchase,,,,,
0,81604,63704,117798,288213,996369
1,106123,76853,165893,376301,2554415


In [21]:
meta_df.head(5)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Digital Music,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,12.05,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SWV Format: Audio CD,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000002X4C,0,0,0
1,Digital Music,Rio Angie,5.0,1,,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...",14.98,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Shrimp City Slim (Artist) Format: Audio CD,"CDs & Vinyl, Jazz, Avant Garde & Free Jazz","{""Product Dimensions"": ""5.6 x 0.4 x 4.9 inches...",B00902T10Y,0,0,0
2,Digital Music,Lost in Love,5.0,9,,,24.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Nastyboy Klick Format: Audio CD,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore","{""Package Dimensions"": ""4.7 x 4.6 x 0.1 inches...",B00000DALY,0,0,0
3,Digital Music,Somewhere in Time,4.8,1186,,The 1980 soundtrack to the now classic motion ...,11.55,"{'hi_res': [None, None], 'large': ['https://m....","{'title': [], 'url': [], 'user_id': []}","John Barry (Composer), Barry, John (Comp...","CDs & Vinyl, Soundtracks, Movie Scores","{""Is Discontinued By Manufacturer"": ""No"", ""Lan...",B0000086D1,0,0,0
4,Digital Music,Kimmon Waldruff,5.0,1,,Solo acoustic fingerstyle guitar.,14.07,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Kimmon Waldruff (Artist) Format: Audio CD,"CDs & Vinyl, Folk","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000S6W7KC,0,0,0


In [19]:
meta_df['main_category'].unique()

array(['Digital Music', 'Movies & TV', 'Cell Phones & Accessories',
       'Books', 'Musical Instruments', 'Tools & Home Improvement',
       'Amazon Home', 'All Electronics', 'Toys & Games',
       'Health & Personal Care', 'Office Products', 0, 'Software',
       'Home Audio & Theater', 'Video Games', 'Audible Audiobooks',
       'Industrial & Scientific', 'Grocery', 'All Beauty',
       'Arts, Crafts & Sewing', 'Sports & Outdoors', 'Pet Supplies',
       'Baby', 'Camera & Photo', 'Appliances', 'Computers',
       'AMAZON FASHION', 'Collectible Coins', 'Entertainment',
       'Automotive'], dtype=object)

In [20]:
meta_df['categories'].unique()

array(['CDs & Vinyl, Dance & Electronic, House',
       'CDs & Vinyl, Jazz, Avant Garde & Free Jazz',
       'CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore',
       'CDs & Vinyl, Soundtracks, Movie Scores', 'CDs & Vinyl, Folk',
       'CDs & Vinyl, International Music, South & Central America, Argentina',
       'CDs & Vinyl, Pop', 'CDs & Vinyl, Jazz, Swing Jazz',
       'CDs & Vinyl, Country', 'CDs & Vinyl, Classical, Chamber Music',
       'CDs & Vinyl, Rock, Progressive, Progressive Rock',
       'CDs & Vinyl, Pop, Oldies, Traditional Pop',
       'CDs & Vinyl, International Music, Pacific Islands, Hawaii',
       'CDs & Vinyl, Pop, Oldies', 'CDs & Vinyl, Folk, Contemporary Folk',
       'CDs & Vinyl, Rock, Hard Rock',
       'CDs & Vinyl, Broadway & Vocalists, Musicals',
       'CDs & Vinyl, Jazz, Bebop', 'CDs & Vinyl, Vinyl Store',
       'CDs & Vinyl, Jazz, Modern Postbebop',
       'CDs & Vinyl, Classical, Forms & Genres, Concertos',
       'CDs & Vinyl, Metal', 'CDs & Vinyl, P

In [7]:
# drop cols dont provide contextual meaning
# user_df = user_df.drop(['images'], axis='columns')
# user_df

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,True
1,5.0,Five Stars,LOVE!!,B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,True
2,3.0,Three Stars,Sad there is not the versions with the real/or...,B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,True
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,True
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,False
...,...,...,...,...,...,...,...,...,...
4827268,5.0,good cd,I love this cd and love the movie thank u I ha...,B000002VPH,B000002VPH,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308413175000,0,True
4827269,5.0,hot cd,I love the cd it play real well and was delive...,B000084T18,B000084T18,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308412875000,0,True
4827270,5.0,Superb sounding remaster,Such a great remaster you can fully appreciate...,B004OFWLO0,B004OFWLO0,AHRJPHOI5JHYEQVSDMNX6736QH3Q,1505425559120,1,True
4827271,1.0,"Very, very disappointing.",What this CD is lacking is a heart. The music...,B000GIXIAK,B000GIXIAK,AH4PJ73QN75AJM5VSCT53AOADCGA,1157470110000,6,False


In [8]:
# drop cols dont provide contextual meaning
meta_df = meta_df.drop(['main_category', 'images', 'videos', 'store', 'price', 'bought_together', 'subtitle', 'author'], axis='columns')

In [9]:
# merge df vs meta
merged_df = user_df.merge(meta_df, on='parent_asin', how='right', suffixes=('', '_meta'))
merged_df = merged_df.rename(columns={'title': 'title_review'})
merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
merged_df.head(5)

,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_meta,average_rating,rating_number,features,description,categories,details
0,1.0,Weak,Definitely the weakest of the three albums SWV...,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,2015-01-03 06:22:17,1.0,False,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
1,1.0,Too Many Guest Spots,Unlike It's About Time and New Beginning this ...,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,2013-09-16 20:19:59,1.0,True,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
2,4.0,"When &quot;Tension&quot; is released, expect q...","Their third and final album as a group, &quot;...",B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,2000-03-19 00:44:40,1.0,False,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
3,5.0,Thanks,thank you,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,2015-03-27 18:15:21,0.0,True,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
4,2.0,Release Some Tension - 2.5 stars,When I was trying to decide whether or not to ...,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,2010-11-13 08:44:58,3.0,False,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."


In [10]:
# filter merge_df with rating >= 3
merged_df['like'] = merged_df['rating'] > 3

# data.drop_duplicates(subset=['asin'])

In [11]:
# encode user_id and asin
merged_df['user_id_encoded'] = LabelEncoder().fit_transform(merged_df['user_id'])
merged_df['asin_encoded'] = LabelEncoder().fit_transform(merged_df['asin'])
merged_df['parent_asin_encoded'] = LabelEncoder().fit_transform(merged_df['parent_asin'])

# data partition
# merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)
df_train, df_test = train_test_split(merged_df, test_size=0.2, random_state=42)
train_interactions_set = set(zip(df_train['user_id_encoded'], df_train['asin_encoded']))
df_test = df_test[~df_test[['user_id_encoded', 'asin_encoded']].apply(tuple, axis=1).isin(train_interactions_set)]

In [12]:
# lightfm
dataset = Dataset()
dataset.fit(users= merged_df['user_id_encoded'], items = merged_df['asin_encoded'])
train_interactions, train_w = dataset.build_interactions((row['user_id_encoded'], row['asin_encoded']) for id, row in df_train.iterrows())
test_interactions, test_w = dataset.build_interactions((row['user_id_encoded'], row['asin_encoded']) for id, row in df_test.iterrows())

NameError: name 'Dataset' is not defined

In [ ]:
model = LightFM(loss='warp', item_alpha=1e-6, user_alpha=1e-6)
# train model
model.fit(train_interactions, epochs=50, num_threads=2, sample_weight=train_w)

In [ ]:
def add_combined_text(row):
    combined_text = f"title_review: {str(row['title_review'])}, text: {str(row['text'])}, title_meta: {str(row['title_meta'])}, features: {str(row['features'])}, description: {str(row['description'])}, details: {str(row['details'])}, categories: {str(row['categories'])}"
    row['combined_text'] = combined_text
    return row

# data = merged_df.map(add_combined_text)
merged_df = merged_df.apply(add_combined_text, axis=1)
# combined_texts = merged_df["combined_text"]
merged_df

,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_meta,...,rating_number,features,description,categories,details,like,user_id_encoded,asin_encoded,parent_asin_encoded,combined_text
0,3.0,Three Stars,all advertizements,B00FA7T630,B00FA7T630,AFNRZBYAPLFKJNJU5BK3TTRWHJ6A,2015-10-26 17:49:26.000,0,1,GQ Print Access Print Magazine,...,10,,"Product Description, Dive into, GQ, ’s culture...",,"{""Date First Available"": ""June 2, 2020"", ""Manu...",False,24328,2499,2499,"title_review: Three Stars, text: all advertize..."
1,5.0,A great audiophile audio magazine that covers ...,A great audiophile audio magazine that covers ...,B00F8P62PO,B00F8P62PO,AETSCPCEEZ3S3BGNU4WCIN4H62ZQ,2018-01-11 07:17:33.014,6,1,Hi-Fi + Print Magazine,...,44,,Hi-Fi+ is Europe's premier English-language hi...,"Magazine Subscriptions, Arts, Music & Photogra...","{""Date First Available"": ""September 19, 2013"",...",True,12133,2498,2498,title_review: A great audiophile audio magazin...
2,3.0,inspirational,i really like the ideas they have in this maga...,B003F1W9T6,B003F1W9T6,AGQCHHMSO6S2DJLWGGFLCVUY477Q,2011-12-24 03:36:48.000,2,1,Paper Crafts,...,3,,,,"{""Date First Available"": ""May 12, 2021""}",False,40505,2095,2095,"title_review: inspirational, text: i really li..."
3,2.0,Should be called card making,"I ordered this magazine because I enjoy ""paper...",B003F1W9T6,B003F1W9T6,AEIKS6KU32T7VBIAP7YIMR2LICQQ,2013-03-03 05:28:59.000,0,1,Paper Crafts,...,3,,,,"{""Date First Available"": ""May 12, 2021""}",False,6799,2095,2095,"title_review: Should be called card making, te..."
4,4.0,Four Stars,Son loves this,B00007AXX1,B00007AXX1,AGDDHCNIGZPKBQRST6UVTCVPD53A,2015-12-04 12:24:30.000,0,1,Horse Illustrated,...,284,,,"Magazine Subscriptions, Sports, Recreation & O...","{""Package Dimensions"": ""10.79 x 8.11 x 0.31 in...",True,34411,871,871,"title_review: Four Stars, text: Son loves this..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71492,1.0,"Beautiful fashion magazine, subscription never...",I have been subscribing to V for a few years n...,B0007INIGI,B0007INIGI,AFDU3G7WMDWWA2XJLYNQLHSIEZMQ,2012-03-01 16:47:32.000,1,1,V Magazine - Ny Print Magazine,...,5,,"V Magazine, was launched in September 1999 as ...","Magazine Subscriptions, Fashion & Style, Men","{""Date First Available"": ""September 14, 2006"",...",False,19700,1315,1315,"title_review: Beautiful fashion magazine, subs..."
71493,1.0,What a fiasco,I subscribed to this magazine a year ago and n...,B0002800W8,B0002800W8,AG5BWPCEIS6FEJKQZTLJVQLGQNGA,2007-09-05 15:08:34.000,2,0,Victorian Review Print Magazine,...,1,,"Published since 1972,, Victorian Review, is an...",,"{""Date First Available"": ""September 14, 2006"",...",False,31564,1248,1248,"title_review: What a fiasco, text: I subscribe..."
71494,1.0,Have yet to receive the first issue,It has been a month since I purchased this mag...,B00007L0RC,B00007L0RC,AHA6K63RE5ODXSTDNPRND73U3EMQ,2011-02-15 16:55:54.000,0,1,Visto Print Magazine,...,2,,"A news magazine targeted at the wider public, ...",,"{""Date First Available"": ""September 15, 2013"",...",False,47920,1032,1032,title_review: Have yet to receive the first is...
71495,4.0,Super Fast Delivery,I was totally surprised how fast I received th...,B00007L0RC,B00007L0RC,AGQT3FF5NHZUOS3YSEAMIRY65VRQ,2012-01-18 22:58:43.000,0,1,Visto Print Magazine,...,2,,"A news magazine targeted at the wider public, ...",,"{""Date First Available"": ""September 15, 2013"",...",True,40744,1032,1032,"title_review: Super Fast Delivery, text: I was..."


In [ ]:
def get_purchase_history(merged_df, user_id, k):
    user_hist = merged_df[merged_df['user_id_encoded']==user_id]
    user_hist_sorted = user_hist.sort_values(by='timestamp', ascending=False)
    recent_purchase = user_hist_sorted[['asin', 'title_meta', 'description', 'combined_text']].head(k)
    return recent_purchase

def get_like_items(merged_df, user_id):
    like_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==True)].sort_values(by='timestamp', ascending=False)
    like_items = like_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

def get_dislike_items(merged_df, user_id):
    dislike_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==False)].sort_values(by='timestamp', ascending=False)
    dislike_items = dislike_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

In [ ]:
def evaluate_recall(model, train_interactions, test_interactions, k):
    precision= precision_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    recall= recall_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    auc= auc_score(model, test_interactions, train_interactions=train_interactions).mean()
    return precision, recall, auc

eval_metrics = evaluate_recall(model, train_interactions, test_interactions, k=50)

In [ ]:
print(eval_metrics)

(0.010412165, 0.20115773521154753, 0.85380054)


In [ ]:
def get_top_k(model, dataset, merged_df, user_id, k):
    n_users, n_items = dataset.interactions_shape()

    user_ids = np.full(n_items, user_id)  # or use np.repeat(user_id, n_items)
    # item_ids = np.arange(n_items)
    interacted_items = set(merged_df[merged_df['user_id_encoded']==user_id]['asin_encoded'])

    scores = model.predict(user_ids, np.arange(n_items))
    scores[list(interacted_items)] = -np.inf

    top_items_encoded = np.argsort(-scores)[:k]
    asin_info = dict(zip(merged_df['asin_encoded'], zip(merged_df['asin'], merged_df['title_meta'], merged_df['combined_text'])))
    recs = [asin_info[item] for item in top_items_encoded if item in asin_info]
    return recs

# user_id_encoder = dict(zip(merged_df['user_id_encoded'], merged_df['user_id']))
recs_dict = {user_id: get_top_k(model, dataset, merged_df, user_id, k=50) for user_id in merged_df['user_id_encoded'].unique()}

def get_recs(user_id):
    return recs_dict.get(user_id, [])

In [ ]:
user_id_test = merged_df['user_id_encoded'].iloc[8]
recent_purchases = get_purchase_history(merged_df, user_id_test, k=10)
like_items = get_like_items(merged_df, user_id_test)
dislike_items = get_dislike_items(merged_df, user_id_test)
top_k_recs= get_recs(user_id_test)

In [ ]:
top_k_recs

[('B00007B1JS',
  'Soap Opera Digest',
  'title_review: Subscription, text: I am unable to review this magazine  I have never received an issue.<br /><br />Needless to say, I am not happy about this.<br /><br />Lynda Epler, title_meta: Soap Opera Digest, features: , description: , details: {"Date First Available": "April 30, 2015"}, categories: Magazine Subscriptions, Entertainment & Pop Culture, Television'),
 ('B00GBJIEQG',
  'Inked',
  'title_review: Five Stars, text: Great magazine, title_meta: Inked, features: , description: , details: {"Date First Available": "October 31, 2013"}, categories: Magazine Subscriptions, Professional & Educational Journals, Professional & Trade, Arts, Photography'),
 ('B00007EN1O',
  'Alpaca    Print Magazine',
  'title_review: Excellent publication about alpacas, text: This magazine is a must for any alpaca owner, would be alpaca owner, or one who is just learning about these beautiful animals.<br />It is beautifully done, showing pictures of alpacas,

In [ ]:
user_id_test = merged_df['user_id_encoded'].iloc[8]
recent_purchases = get_purchase_history(merged_df, user_id_test, k=3)
recent_purchases

,asin,title_meta,description,combined_text
8,B00007AXX1,Horse Illustrated,,"title_review: Great Magazine, text: This magaz..."


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 6.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.document_loaders import TextLoader
from langchain.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:

# since max number of tokens of sentence transformers is 512
# split the text into smaller chunks that fit within this limit before embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# prepare documents
def prepare_documents(user_id_test, recs):
    documents = []
    for asin, title_meta, combined_text in recs:
        chunks = text_splitter.split_text(combined_text)
        for chunk in chunks:
            metadata = {
                "asin": asin,
                "title_meta": title_meta,
            }
            doc = Document(page_content=chunk, metadata=metadata)
            documents.append(doc)
    return documents

user_documents = prepare_documents(user_id_test, top_k_recs)
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN_HERE"
# YOUR_HUGGINGFACE_TOKEN_HERE
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
chroma_client = Chroma.from_documents(documents=user_documents, embedding=embeddings, persist_directory="./content/chroma_db")

def get_search_result(user_query, chroma_client, k):
    res = chroma_client.similarity_search(query=user_query, k=k)
    return res



<ipython-input-25-0d67646e6ac0>:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
user_query= "beauty magazine that can provide beauty tips"
retrieved_docs = get_search_result(user_query, chroma_client, k=5)
retrieved_docs

[Document(metadata={'asin': 'B00005QJE2', 'title_meta': 'Nylon    Print Magazine'}, page_content='interested in a fashion magazine that goes beyond a "beach read.", title_meta: Nylon    Print Magazine, features: , description: A vibrant and proactive voice for today\'s hip, intelligent, young women seeking fresh perspectives on fashion, beauty and music., details: {"Date First Available": "September 14, 2006", "Manufacturer": "Nylon Media Inc."}, categories: Magazine Subscriptions, Business & Investing, International'),
 Document(metadata={'asin': 'B00005QJE2', 'title_meta': 'Nylon    Print Magazine'}, page_content='title_review: inspiring, text: as a high school student aspiring to attend art school, nylon has provided me with countless sources of inspiration in their creative photography, layout, and decoration. i look forward to reading the magazine each month, and i honestly devour every last word and picture.<br /><br />nylon is a creative, whimsical and fantastic magazine. i high

In [ ]:
# !pip install accelerate

In [ ]:
model_id = "nvidia/Mistral-NeMo-Minitron-8B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# dtype = torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(model_id, load_in_8bit=True)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
)

# HuggingFace pipeline integration
hf = HuggingFacePipeline(pipeline=pipe)
llm = hf

tokenizer_config.json:   0%|          | 0.00/177k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
prompt_template = """
Imagine you are a recommender. Given the user's recent purchase history, liked items, disliked items, and query:
User Query: {question}
Recent Purchases: {purchase_history}
Liked Items: {liked_items}
Disliked Items: {disliked_items}

Please rank the following items based on relevance and provide a brief explanation for each:
{context}
Reranked Recommendations:
"""

def personal_rerank(user_query, recent_purchase, retrieved_docs, like_items, dislike_items, llm, prompt_template):
    purchase_hist = "\n".join([f"{i+1}. {row['title_meta']}: {row['description']}" for i, (_, row) in enumerate(recent_purchases.iterrows())])
    context = "\n".join(f"{doc.metadata['title_meta']}: {doc.page_content}" for doc in retrieved_docs)
    prompt = prompt_template.format(
        question=user_query,
        purchase_history=purchase_hist,
        liked_items="\n".join(like_items),
        disliked_items="\n".join(dislike_items),
        context=context
    )
    return llm(prompt)


In [ ]:
user_query = "beauty magazine that can provide beauty tips"
retrieved_docs = get_search_result(user_query, chroma_client, k=5)
